# Проект 1. Аналитическая записка — PRIME

**Автор:** Троценко Анастасия Александровна, группа 262.

## Цель работы

В работе проверяются две бизнес-гипотезы на данных сервиса PRIME. Для каждого исследования заранее указаны бизнес-вопрос, проверяемая гипотеза, единица наблюдения (зерно), период и способ проверки. Данные загружаются из базы проекта; расчёты метрик, статистическая проверка, факторная декомпозиция и графики выполняются в Python.

**Исследование 1:** связаны ли дополнительные участники подписки с долей завершения подписки?

**Исследование 2:** внесло ли повышение тарифных цен положительный вклад в рост выручки в марте 2026 года по сравнению с февралем?



## 0. Импорты

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


Подключены библиотеки для работы с данными, SQL-запросами и графиками. `pandas` помогает собирать и сравнивать таблицы, `numpy` — выполнять расчёты, `matplotlib` и `seaborn` — строить графики, а `sqlalchemy` — обращаться к базе данных. Здесь ещё нет бизнес-результата: это подготовка инструментов.

## 1. Подключение к базе

In [ ]:
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден. Запустите Jupyter из корня prime-monitor "
        "через `uv run jupyter lab` или выберите ядро .venv."
    ) from None

engine = get_engine()
def q(sql: str, **params) -> pd.DataFrame:
    with engine.connect() as conn:
        conn.execute(text("SET TIME ZONE 'UTC'"))
        return pd.read_sql(text(sql), conn, params=params)

print("Подключение успешно:", q("SELECT current_user AS login").iloc[0, 0])


## Исследование 1. Связаны ли дополнительные участники с завершением подписки?

**Бизнес-вопрос:** отличается ли доля подписок, завершившихся в анализируемом периоде, между подписками с дополнительными участниками и подписками без них?

**Гипотеза H1:** доля завершения ниже у подписок с дополнительными участниками, чем у подписок без дополнительных участников.

**Нулевая гипотеза H0:** доли завершения в двух группах одинаковы. **Альтернативная гипотеза H1:** доля завершения у подписок с участниками ниже.

**Зерно расчёта:** одна строка — одна подписка (`subscription_id`). Таблица участников предварительно агрегируется до уровня подписки; это предотвращает повторение одной подписки и двойной счёт. Группы определяются по наличию дополнительных участников.

**Метрика:** доля подписок, завершившихся в анализируемом периоде, среди всех подписок соответствующей группы. Сравниваются доли, а не абсолютные количества, поскольку размеры групп могут отличаться.

**Период:** 1 января — 30 июня 2026 года. После расчёта долей используется двусторонний Z-тест для сравнения пропорций при уровне значимости 0,05. Гипотеза получает статистическую поддержку только при результате, совместимом с направлением гипотезы и статистической значимостью.


In [ ]:
START = pd.Timestamp("2026-01-01", tz="UTC")
END = pd.Timestamp("2026-07-01", tz="UTC")
subs = q("""
    SELECT subscription_id, owner_client_id, tariff_id, started_at, ended_at,
           is_trial, cancel_reason
    FROM prime.subscriptions
    WHERE started_at < :start
      AND (ended_at IS NULL OR ended_at >= :start)
""", start=START.to_pydatetime())
members = q("""
    SELECT subscription_id, client_id, joined_at, left_at
    FROM prime.subscription_members
    WHERE joined_at < :start
      AND (left_at IS NULL OR left_at >= :start)
""", start=START.to_pydatetime())
tariffs = q("""
    SELECT tariff_id, tariff_name, monthly_fee, cashback_rate, share_slots,
           valid_from, valid_to
    FROM prime.tariffs
""")


После загрузки таблиц дальнейшая подготовка выполняется в Python. Число участников агрегируется до одной строки на подписку, чтобы не задвоить подписки при сравнении групп.

In [ ]:
# Приводим даты к формату datetime и агрегируем участников до уровня подписки.
for col in ["started_at", "ended_at"]:
    subs[col] = pd.to_datetime(subs[col], utc=True, errors="coerce")
for col in ["joined_at", "left_at"]:
    members[col] = pd.to_datetime(members[col], utc=True, errors="coerce")
member_counts = (
    members.groupby("subscription_id")["client_id"]
    .nunique().rename("n_members").reset_index()
)
analysis = subs.merge(member_counts, on="subscription_id", how="left", validate="1:1")
analysis["n_members"] = analysis["n_members"].fillna(0).astype(int)
analysis["has_members"] = analysis["n_members"].gt(0)
analysis["cancelled_in_period"] = (
    analysis["ended_at"].notna()
    & analysis["ended_at"].ge(START)
    & analysis["ended_at"].lt(END)
)
analysis["cancel_month"] = analysis["ended_at"].dt.to_period("M").astype("string")
analysis.loc[~analysis["cancelled_in_period"], "cancel_month"] = pd.NA
print(f"Подписок в когорте: {analysis['subscription_id'].nunique():,}")
print(f"С дополнительными участниками: {analysis['has_members'].sum():,}")
print(f"Без дополнительных участников: {(~analysis['has_members']).sum():,}")
display(analysis.head())


Данные участников сначала приводятся к уровню подписки, чтобы одна подписка не повторялась несколько раз из-за нескольких дополнительных участников. Это важно: если напрямую соединить платежи с таблицей участников, одна и та же выручка может посчитаться несколько раз. В первой гипотезе единицей анализа является подписка, а не отдельный участник.

### Проверка качества данных

In [ ]:
quality = {
    "rows": len(analysis),
    "unique_subscription_id": analysis["subscription_id"].nunique(),
    "duplicate_subscription_id_rows": int(analysis["subscription_id"].duplicated().sum()),
    "memberships_without_subscription_in_cohort": int((~members["subscription_id"].isin(analysis["subscription_id"])).sum()),
    "missing_started_at": int(analysis["started_at"].isna().sum()),
    "ended_before_start": int((analysis["ended_at"].notna() & (analysis["ended_at"] < analysis["started_at"])).sum()),
}
display(pd.Series(quality, name="value").to_frame())
assert quality["rows"] == quality["unique_subscription_id"], "Нарушено зерно: не одна строка на подписку."
assert quality["duplicate_subscription_id_rows"] == 0


**Проверка качества данных.** Здесь проверяется, что после подготовки одна строка действительно представляет одно наблюдение на выбранном уровне анализа, а нужные поля заполнены. Эти проверки нужны, чтобы не принять дубликаты или пропуски за реальный бизнес-эффект. Если проверка обнаруживает проблему, её нужно разобраться до интерпретации результата.

### Основной результат: доля отмен

In [ ]:
summary = (
    analysis.groupby("has_members", observed=True)
    .agg(subscriptions=("subscription_id", "nunique"),
         cancellations=("cancelled_in_period", "sum"),
         cancellation_rate=("cancelled_in_period", "mean"),
         median_members=("n_members", "median"))
    .rename(index={False: "Без участников", True: "С участниками"})
)
display(summary)
g0 = analysis.loc[~analysis["has_members"], "cancelled_in_period"].astype(int)
g1 = analysis.loc[analysis["has_members"], "cancelled_in_period"].astype(int)
n0, n1 = len(g0), len(g1)
p0, p1 = g0.mean(), g1.mean()
difference = p1 - p0
pooled = (g0.sum() + g1.sum()) / (n0 + n1)
se_test = np.sqrt(pooled * (1 - pooled) * (1/n0 + 1/n1))
z_stat = (p1 - p0) / se_test if se_test > 0 else np.nan
p_value = math.erfc(abs(z_stat) / math.sqrt(2)) if np.isfinite(z_stat) else np.nan
se_ci = np.sqrt(p0*(1-p0)/n0 + p1*(1-p1)/n1)
ci_low, ci_high = difference - 1.96*se_ci, difference + 1.96*se_ci
print(f"Отмена без участников: {p0:.2%} (n={n0:,})")
print(f"Отмена с участниками: {p1:.2%} (n={n1:,})")
print(f"Разница (с участниками − без): {difference:.2%}")
print(f"95% ДИ разницы: [{ci_low:.2%}; {ci_high:.2%}]")
print(f"z = {z_stat:.3f}; p-value = {p_value:.6g}")


**Вывод** Здесь формируются группы подписок: с дополнительными участниками и без них, а затем рассчитывается доля подписок, завершившихся в выбранном периоде. Сравнивать нужно именно доли, а не только абсолютное число отмен: группы могут различаться по размеру. 

### График 1. Доля отмен в двух группах

In [ ]:
plot_data = pd.DataFrame({
    "Группа": ["Без участников", "С участниками"],
    "Доля отмен": [p0, p1],
    "Число подписок": [n0, n1],
})
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=plot_data, x="Группа", y="Доля отмен", ax=ax)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set(title="Доля отмен за январь–июнь 2026 года", xlabel="", ylabel="Доля отмен")
for i, row in plot_data.iterrows():
    ax.text(i, row["Доля отмен"], f'{row["Доля отмен"]:.1%}\nn={row["Число подписок"]:,}',
            ha="center", va="bottom")
plt.tight_layout()
plt.show()


### Проверка различий по тарифам

In [ ]:
tariff_summary = (
    analysis.merge(tariffs[["tariff_id", "tariff_name"]].drop_duplicates("tariff_id"),
                   on="tariff_id", how="left", validate="m:1")
    .groupby(["tariff_name", "has_members"], dropna=False)
    .agg(n=("subscription_id", "nunique"), cancel_rate=("cancelled_in_period", "mean"))
    .reset_index()
)
display(tariff_summary)


### Вывод по первой гипотезе

Доля завершений подписок без дополнительных участников — **21,14%**, с участниками — **21,10%**. Разница составляет около **0,04 процентного пункта**, то есть практически отсутствует. Двусторонний Z-тест дал `p-value = 0,8407`, что больше выбранного уровня значимости **0,05**. Поэтому мы **не нашли статистически значимых свидетельств**, что подписки с дополнительными участниками завершаются реже. Это не доказывает полного отсутствия эффекта во всех ситуациях, но на имеющихся данных гипотеза не подтверждается.

## Исследование 2. Внесло ли повышение тарифных цен вклад в рост выручки?

**Бизнес-вопрос:** насколько изменилась выручка PRIME с февраля на март 2026 года и согласуется ли изменение с повышением тарифных цен?

**Гипотеза H2:** повышение цен подписок PRIME внесло положительный вклад в увеличение выручки в марте 2026 года по сравнению с февралем.

Какие таблицы используем:
- prime.payments — таблица платежей. Из неё берём дату, сумму и статус платежа. Для расчёта выручки учитываем только успешные платежи со статусом success.
- prime.subscriptions — связывает платежи с подписками и позволяет определить тариф, к которому относится платёж.
- prime.tariffs — содержит названия тарифов, месячные цены и интервалы действия цен. Используется для сравнения цен февраля и марта и расчёта контрфактического сценария

**Зерно исходных данных:** одна строка таблицы платежей - один платёж. В расчёт выручки входят только платежи со статусом `success`; затем данные агрегируются до календарного месяца. Для анализа по тарифам платёж связывается с подпиской и тарифом. Такое зерно выбрано потому, что выручка - сумма денежных поступлений, а не число подписок или клиентов.

**Ключевые метрики:** выручка, количество успешных платежей, средняя сумма успешного платежа и тарифная цена, действовавшая в соответствующий период.

**Метод:** сначала сравнивается фактическая выручка февраля и марта, затем прирост раскладывается симметричным методом на вклад количества успешных платежей и вклад средней суммы платежа. История цен и контрфактический сценарий при февральских ценах используются как дополнительная проверка. Контрфактический сценарий зависит от корректности истории тарифов и не является экспериментальным доказательством причинности.


### Шаг 1. Общая выручка по месяцам

**Промежуточный вывод.** Выручка выросла на **14,73 млн ₽ (+33,86%)**: с 43,50 млн ₽ в феврале до 58,23 млн ₽ в марте. Сам факт роста подтверждён расчётом и графиком; теперь нужно проверить, могло ли повышение тарифных цен внести вклад в это изменение.

In [ ]:
REVENUE_START = pd.Timestamp("2026-02-01")
REVENUE_END = pd.Timestamp("2026-04-01")
revenue = q("""
    SELECT DATE_TRUNC('month', p.paid_at)::date AS month,
           COUNT(*) AS successful_payments,
           SUM(p.amount) AS revenue,
           AVG(p.amount) AS avg_payment
    FROM prime.payments p
    WHERE p.status = 'success'
      AND p.paid_at >= :start AND p.paid_at < :end
    GROUP BY 1 ORDER BY 1
""", start=REVENUE_START.to_pydatetime(), end=REVENUE_END.to_pydatetime())
revenue["month"] = pd.to_datetime(revenue["month"])
for col in ["successful_payments", "revenue", "avg_payment"]:
    revenue[col] = pd.to_numeric(revenue[col])
display(revenue.round(2))

expected_months = pd.to_datetime(["2026-02-01", "2026-03-01"])
if len(revenue) != 2 or not set(expected_months).issubset(set(revenue["month"])):
    raise ValueError("Ожидались данные за февраль и март 2026 года.")


### График 2. Выручка PRIME по месяцам

In [ ]:
# Строим график из результата SQL-запроса, а не из вручную заданных чисел.
revenue_plot = revenue.copy()
revenue_plot["Месяц"] = revenue_plot["month"].dt.strftime("%B %Y")
revenue_plot["Выручка, млн ₽"] = revenue_plot["revenue"] / 1_000_000
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(revenue_plot["Месяц"], revenue_plot["Выручка, млн ₽"])
ax.bar_label(bars, labels=[f"{v:.2f} млн ₽" for v in revenue_plot["Выручка, млн ₽"]], padding=4)
ax.set_title("Мы обнаружили рост выручки PRIME в марте 2026 года")
ax.set_ylabel("Выручка, млн ₽")
ax.set_xlabel("Месяц")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


**Что показывает график.** Выручка рассчитана запросом из базы по успешным платежам и затем отображена по месяцам. В исходном анализе февральская выручка составила **43,50 млн ₽**, мартовская — **58,23 млн ₽**: прирост **14,73 млн ₽ (+33,86%)**. График показывает обнаруженное изменение, а следующие шаги проверяют, какие факторы могли с ним совпасть. Если актуальные данные базы отличаются, подписи на графике автоматически изменятся.

**Что считается выручкой.** На этом шаге выбираются успешные платежи и агрегируются по календарному месяцу. Успешный платёж включается в выручку, а платежи с другими статусами исключаются. Месячный уровень выбран потому, что бизнес-вопрос касается изменения выручки между февралем и мартом 2026 года; для последующей декомпозиции дополнительно понадобятся количество платежей и их средняя сумма.

### Шаг 2. Симметричная факторная декомпозиция

**Результат факторной декомпозиции из исходного выполненного анализа:**

- Вклад увеличения числа успешных платежей — примерно **+6,29 млн ₽** (42,7% общего прироста).
- Вклад изменения средней суммы платежа — примерно **+8,43 млн ₽** (57,3% общего прироста).

Сумма вкладов равна общему приросту около **14,73 млн ₽**. На этом шаге видно, что фактор средней суммы платежа арифметически важнее фактора количества платежей. Но это ещё не чистый эффект цены: средняя сумма может зависеть от скидок и состава платежей.

In [ ]:
feb = revenue.loc[revenue["month"] == pd.Timestamp("2026-02-01")].iloc[0]
mar = revenue.loc[revenue["month"] == pd.Timestamp("2026-03-01")].iloc[0]

R_feb, R_mar = float(feb["revenue"]), float(mar["revenue"])
N_feb, N_mar = int(feb["successful_payments"]), int(mar["successful_payments"])
A_feb, A_mar = float(feb["avg_payment"]), float(mar["avg_payment"])
delta_revenue = R_mar - R_feb
volume_effect = (N_mar - N_feb) * (A_feb + A_mar) / 2
average_effect = (A_mar - A_feb) * (N_feb + N_mar) / 2
print(f"Выручка февраля: {R_feb:,.2f} ₽")
print(f"Выручка марта: {R_mar:,.2f} ₽")
print(f"Прирост выручки: {delta_revenue:,.2f} ₽ ({delta_revenue/R_feb:.1%})")
print(f"Вклад количества успешных платежей: {volume_effect:,.2f} ₽")
print(f"Вклад средней суммы платежа: {average_effect:,.2f} ₽")
print(f"Доля вклада количества: {volume_effect/delta_revenue:.1%}")
print(f"Доля вклада средней суммы: {average_effect/delta_revenue:.1%}")
assert np.isclose(volume_effect + average_effect, delta_revenue, atol=0.02)


 Помимо общей выручки, расчёт собирает число успешных платежей и среднюю сумму одного платежа. Это две составляющие выручки: в упрощённом виде выручка равна количеству успешных платежей, умноженному на среднюю сумму платежа. Такое представление позволяет перейти от фиксации роста к объяснению его арифметических источников.

### График 3. Вклад факторов в прирост выручки

In [ ]:
effects = pd.DataFrame({
    "Фактор": ["Количество успешных платежей", "Средняя сумма платежа"],
    "Вклад, ₽": [volume_effect, average_effect]
})
fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.barplot(data=effects, x="Фактор", y="Вклад, ₽", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f} млн"))
ax.set(title="Симметричная декомпозиция: февраль → март",
       xlabel="", ylabel="Вклад в изменение выручки")
plt.xticks(rotation=10, ha="right")
plt.tight_layout()
plt.show()


**Результат факторной декомпозиции.** Общий прирост выручки около **14,73 млн ₽** распределён между двумя факторами: число успешных платежей внесло около **+6,29 млн ₽ (42,7%)**, а средняя сумма платежа — около **+8,43 млн ₽ (57,3%)**. Вклады складываются в общий прирост с учётом округления. Средняя сумма платежа объясняет большую часть изменения в этой арифметической декомпозиции, но она не равна тарифной цене и может меняться по другим причинам.

### Шаг 3. История тарифов и фактические суммы

In [ ]:
tariff_history = q("""
    SELECT tariff_id, tariff_name, monthly_fee, valid_from, valid_to
    FROM prime.tariffs
    WHERE valid_from < :end
      AND (valid_to IS NULL OR valid_to >= :start)
    ORDER BY tariff_id, valid_from
""", start=pd.Timestamp("2026-02-01").to_pydatetime(),
     end=pd.Timestamp("2026-04-01").to_pydatetime())
display(tariff_history)


**Что показывает график декомпозиции.** Каждый столбец — расчётный вклад соответствующего фактора в изменение выручки между февралем и мартом. Оба вклада положительные, то есть в рамках модели оба фактора увеличили выручку. Более высокий вклад средней суммы платежа означает, что её изменение было важнее по величине, чем изменение количества платежей; это ещё не доказывает, что весь эффект вызван повышением тарифов.

In [ ]:
tariff_payments = q("""
    SELECT DATE_TRUNC('month', p.paid_at)::date AS month,
           s.tariff_id,
           COUNT(*) AS successful_payments,
           SUM(p.amount) AS revenue,
           AVG(p.amount) AS avg_payment,
           MIN(p.amount) AS min_payment,
           MAX(p.amount) AS max_payment,
           COUNT(DISTINCT p.amount) AS distinct_amounts
    FROM prime.payments p
    JOIN prime.subscriptions s ON p.subscription_id = s.subscription_id
    WHERE p.status = 'success'
      AND p.paid_at >= :start AND p.paid_at < :end
    GROUP BY 1, 2 ORDER BY 1, 2
""", start=pd.Timestamp("2026-02-01").to_pydatetime(),
     end=pd.Timestamp("2026-04-01").to_pydatetime())
tariff_payments["month"] = pd.to_datetime(tariff_payments["month"])
display(tariff_payments.round(2))


Фактическая сумма платежа не обязательно совпадает с установленной месячной ценой: могут существовать разные условия оплаты, скидки или другие особенности данных. Поэтому сначала сопоставляются платежи, тарифы и даты действия цен. Если одно и то же наблюдение подходит сразу к нескольким записям истории тарифов, вывод о цене может быть искажён.

Таблица выше позволяет проверить, совпадают ли фактические суммы с тарифными суммами, и есть ли разные значения в одной группе месяц × тариф.

### Шаг 4. Декомпозиция по тарифам

**Как интерпретировать сравнение по тарифам.** Разбиение по тарифам позволяет проверить, одинаково ли менялись количество успешных платежей и средняя сумма платежа в каждой тарифной группе. Суммировать эффекты по тарифам можно только при корректном сопоставлении каждой оплаты с одним тарифом и одной действующей на дату оплаты ценой. Если история тарифов неоднозначна, этот график следует считать диагностическим, а не окончательным доказательством.

In [ ]:

# Сводим данные по тарифу и месяцу в одну таблицу.
# tariff_payments должна быть рассчитана ранее из успешных платежей.
tariff_monthly = tariff_payments.copy()

# Оставляем только февраль и март 2026 года.
tariff_monthly["month"] = pd.to_datetime(tariff_monthly["month"])
tariff_monthly = tariff_monthly[
    tariff_monthly["month"].dt.strftime("%Y-%m").isin(["2026-02", "2026-03"])
].copy()

# Проверяем, есть ли данные по обоим месяцам для каждого тарифа.
tariff_monthly["month_label"] = tariff_monthly["month"].dt.strftime("%Y-%m")

# Показываем исходные метрики по тарифам.
display(
    tariff_monthly[
        ["tariff_id", "month_label", "successful_payments", "revenue", "avg_payment"]
    ].sort_values(["tariff_id", "month_label"])
)

# Считаем симметричную декомпозицию отдельно по каждому тарифу.
tariff_effects = []

for tariff_id, group in tariff_monthly.groupby("tariff_id"):
    group = group.set_index("month_label")

    # Если одного из месяцев нет, корректно сравнить этот тариф нельзя.
    if not {"2026-02", "2026-03"}.issubset(group.index):
        continue

    feb = group.loc["2026-02"]
    mar = group.loc["2026-03"]

    # Берём количество успешных платежей и среднюю сумму платежа.
    n_feb = feb["successful_payments"]
    n_mar = mar["successful_payments"]
    avg_feb = feb["avg_payment"]
    avg_mar = mar["avg_payment"]

    # Симметрично распределяем изменение выручки между двумя факторами.
    volume_effect = (n_mar - n_feb) * (avg_feb + avg_mar) / 2
    average_effect = (avg_mar - avg_feb) * (n_feb + n_mar) / 2

    tariff_effects.append({
        "tariff_id": tariff_id,
        "revenue_change": mar["revenue"] - feb["revenue"],
        "payment_count_effect": volume_effect,
        "average_payment_effect": average_effect,
    })

tariff_effects = pd.DataFrame(tariff_effects)
display(tariff_effects)


Что это означает:
- На базовом тарифе выручка выросла преимущественно за счёт увеличения средней суммы платежа.
- На расширенном тарифе больший вклад внесло увеличение количества успешных платежей.
- В обоих тарифах оба фактора дали положительный вклад в рост выручки.
- Поэтому рост выручки нельзя объяснить только повышением цены: на него также повлияло изменение числа успешных платежей.
Важное ограничение: декомпозиция показывает, как арифметически складывается прирост выручки, но не доказывает, что именно повышение цены вызвало весь рост средней суммы платежа. Для такого вывода нужно учитывать скидки, состав платежей и корректность истории тарифов.

In [ ]:

# Копируем результаты декомпозиции, чтобы не менять исходную таблицу.
plot_data = tariff_effects.copy()

# Переводим суммы в миллионы рублей для удобства чтения.
plot_data["Количество платежей, млн ₽"] = (
    plot_data["payment_count_effect"] / 1_000_000
)
plot_data["Средняя сумма платежа, млн ₽"] = (
    plot_data["average_payment_effect"] / 1_000_000
)

# Получаем названия тарифов из справочника.
tariff_names = tariffs.drop_duplicates("tariff_id").set_index("tariff_id")["tariff_name"]
plot_data["Тариф"] = plot_data["tariff_id"].map(tariff_names)

# Сортируем тарифы по общему приросту выручки.
plot_data = plot_data.sort_values("revenue_change")

# Строим диаграмму: два фактора показываются рядом для каждого тарифа.
fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(plot_data))
height = 0.32

ax.barh(
    y - height / 2,
    plot_data["Количество платежей, млн ₽"],
    height,
    label="Количество успешных платежей"
)
ax.barh(
    y + height / 2,
    plot_data["Средняя сумма платежа, млн ₽"],
    height,
    label="Средняя сумма платежа"
)

# Подписываем оси и добавляем легенду.
ax.set_yticks(y, plot_data["Тариф"])
ax.set_xlabel("Вклад в изменение выручки, млн ₽")
ax.set_title("Что увеличило выручку по каждому тарифу")
ax.axvline(0, linewidth=1)
ax.legend(frameon=False)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


**Вывод по декомпозиции выручки по тарифам.** В феврале–марте 2026 года выручка выросла по обоим тарифам. Для базового тарифа больший вклад внесло увеличение средней суммы платежа: около 5,24 млн ₽ против 2,92 млн ₽ от количества успешных платежей. Для расширенного тарифа ситуация обратная: вклад количества успешных платежей составил около 3,37 млн ₽, а вклад средней суммы платежа — около 3,20 млн ₽.

Таким образом, факторы роста различаются по тарифам. Для базового тарифа важнее изменение средней суммы платежа, а для расширенного — увеличение количества успешных платежей. Это арифметическое разложение прироста выручки; оно само по себе не доказывает, что изменение тарифных цен стало единственной причиной роста.

### Дополнительная проверка по тарифам

**Проверка истории тарифов.** В этом шаге ищутся пересекающиеся или неоднозначные интервалы действия цен. Для корректного сопоставления у платежа должна определяться одна подходящая запись тарифа на дату платежа. Если таких записей несколько или нет ни одной, контрфактический расчёт нельзя интерпретировать без оговорок: сначала необходимо уточнить историю тарифов.

### Шаг 5. Проверка пересечений в истории тарифов

In [ ]:
tariff_overlaps = q("""
    SELECT a.tariff_id, a.tariff_name,
           a.monthly_fee AS price_1, a.valid_from AS start_1, a.valid_to AS end_1,
           b.monthly_fee AS price_2, b.valid_from AS start_2, b.valid_to AS end_2
    FROM prime.tariffs a
    JOIN prime.tariffs b
      ON a.tariff_id = b.tariff_id
     AND a.valid_from < b.valid_from
     AND (a.valid_to IS NULL OR a.valid_to >= b.valid_from)
    ORDER BY a.tariff_id, a.valid_from, b.valid_from
""")
display(tariff_overlaps)
if tariff_overlaps.empty:
    print("Пересекающиеся интервалы не найдены.")
else:
    print("Найдены пересечения: это ограничение качества истории тарифов, его нужно учитывать в выводе.")


**Зачем нужен контрфактический сценарий.** Он отвечает на условный вопрос: какой была бы расчётная сумма мартовских платежей, если применить к мартовскому объёму февральские тарифные цены? Это сравнение помогает оценить возможный вклад повышения цен, но зависит от допущений о сопоставимости тарифов и платежей. Это не эксперимент и не прямое измерение причинного эффекта.

### Шаг 6. Контрфактический сценарий: мартовское количество платежей при февральских ценах

In [ ]:
# Используем тарифные цены из истории, а не захардкоженные значения.
# Если в феврале у одного тарифа несколько цен, не выбираем одну молча:
# показываем неоднозначность и останавливаем расчёт до проверки справочника.
feb = tariff_payments[tariff_payments["month"] == pd.Timestamp("2026-02-01")].copy()
mar = tariff_payments[tariff_payments["month"] == pd.Timestamp("2026-03-01")].copy()

if feb.empty or mar.empty:
    raise ValueError("Нужны данные по тарифам за оба месяца.")
feb_price_rows = tariff_history[
    (pd.to_datetime(tariff_history["valid_from"]) < pd.Timestamp("2026-03-01"))
    & (tariff_history["valid_to"].isna() | (pd.to_datetime(tariff_history["valid_to"]) >= pd.Timestamp("2026-02-01")))
].copy()
display(feb_price_rows)


Столбцы сравнивают фактическую мартовскую сумму с расчётным сценарием при февральских ценах. Разница между ними - оценка в рамках выбранной модели. Она не означает автоматически, что повышение цен принесло ровно такую дополнительную выручку: результат может зависеть от состава платежей, фактических сумм и качества истории тарифов.

В следующей ячейке февральская цена определяется из истории тарифов на февраль. Если по тарифу найдено несколько возможных цен, расчёт не скрывает проблему и сообщает об неоднозначности.

In [ ]:
feb_price_rows["valid_from"] = pd.to_datetime(feb_price_rows["valid_from"], utc=True, errors="coerce")
feb_price_rows["valid_to"] = pd.to_datetime(feb_price_rows["valid_to"], utc=True, errors="coerce")
feb_price_rows["feb_start"] = pd.Timestamp("2026-02-01", tz="UTC")
feb_price_rows["feb_end"] = pd.Timestamp("2026-03-01", tz="UTC")
feb_active = feb_price_rows[
    (feb_price_rows["valid_from"] < feb_price_rows["feb_end"])
    & (feb_price_rows["valid_to"].isna() | (feb_price_rows["valid_to"] >= feb_price_rows["feb_start"]))
]
price_counts = feb_active.groupby("tariff_id")["monthly_fee"].nunique()
ambiguous = price_counts[price_counts > 1]
if not ambiguous.empty:
    print("ВНИМАНИЕ: несколько тарифных цен на февраль по tariff_id:", ambiguous.to_dict())
    print("Для итогового сценария используем исходный анализ фактических сумм по тарифам; причинный вывод ограничиваем.")
else:
    feb_prices = (feb_active.sort_values("valid_from")
                  .drop_duplicates("tariff_id", keep="last")
                  [["tariff_id", "monthly_fee"]]
                  .rename(columns={"monthly_fee": "february_tariff_price"}))
    march_cf = mar.merge(feb_prices, on="tariff_id", how="left", validate="many_to_one")
    if march_cf["february_tariff_price"].isna().any():
        print("Не удалось сопоставить февральскую цену для всех мартовских тарифов.")
    else:
        march_cf["march_revenue_at_feb_prices"] = march_cf["successful_payments"] * march_cf["february_tariff_price"]
        march_cf["march_tariff_value"] = march_cf["successful_payments"] * march_cf["avg_payment"]
        display(march_cf.round(2))


**Интерпретация тарифной истории.** Сравниваются цены, действовавшие в феврале и марте, по датам начала и окончания действия из базы. Если у тарифа обнаруживаются пересекающиеся интервалы или несколько подходящих цен, результат нужно считать неоднозначным до исправления справочника. Само совпадение повышения цены с ростом выручки не доказывает причинность.


**Промежуточный вывод по второй гипотезе.** В марте выручка оказалась на **14,73 млн ₽** выше февральской. Из этого прироста около **8,43 млн ₽** приходится на изменение средней суммы платежа и около **6,29 млн ₽** — на изменение количества успешных платежей. История тарифов показывает рост цен, а контрфактический сценарий даёт дополнительный аргумент в пользу положительного вклада цены. Однако это расчётное свидетельство, а не окончательное причинное доказательство.

### График 5. Изменение цен тарифов и выручки

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
# Берём названия и цены только из таблицы истории тарифов, загруженной из базы.
tariff_plot = tariff_history.copy()
tariff_plot["valid_from"] = pd.to_datetime(tariff_plot["valid_from"], utc=True, errors="coerce")
tariff_plot["valid_to"] = pd.to_datetime(tariff_plot["valid_to"], utc=True, errors="coerce")
feb_start = pd.Timestamp("2026-02-01", tz="UTC")
mar_start = pd.Timestamp("2026-03-01", tz="UTC")
apr_start = pd.Timestamp("2026-04-01", tz="UTC")
# Находим цену, действовавшую в каждом месяце, по интервалу действия из базы.
feb_rows = tariff_plot[(tariff_plot["valid_from"] < mar_start) & (tariff_plot["valid_to"].isna() | (tariff_plot["valid_to"] >= feb_start))].copy()
mar_rows = tariff_plot[(tariff_plot["valid_from"] < apr_start) & (tariff_plot["valid_to"].isna() | (tariff_plot["valid_to"] >= mar_start))].copy()
feb_prices = feb_rows.sort_values("valid_from").drop_duplicates("tariff_id", keep="last").set_index("tariff_id")["monthly_fee"]
mar_prices = mar_rows.sort_values("valid_from").drop_duplicates("tariff_id", keep="last").set_index("tariff_id")["monthly_fee"]
price_plot = pd.concat([feb_prices.rename("Февраль"), mar_prices.rename("Март")], axis=1).dropna(how="all")
# Названия тарифов берём из справочника; идентификатор остаётся запасным названием при пропуске.
tariff_labels = tariff_plot.drop_duplicates("tariff_id").set_index("tariff_id")["tariff_name"]
price_plot.index = [tariff_labels.get(tid, str(tid)) for tid in price_plot.index]
x = np.arange(len(price_plot))
w = 0.34
b1 = ax.bar(x - w/2, price_plot["Февраль"], w, label="Февраль")
b2 = ax.bar(x + w/2, price_plot["Март"], w, label="Март")
ax.bar_label(b1, fmt="%.0f ₽", padding=3)
ax.bar_label(b2, fmt="%.0f ₽", padding=3)
ax.set_xticks(x, price_plot.index)
ax.set_title("Месячные цены тарифов из базы")
ax.set_ylabel("Цена, ₽")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


На втором графике выручка рассчитывается из таблицы `revenue`, полученной из базы. 

In [ ]:
revenue_plot = revenue[revenue["month"].isin([pd.Timestamp("2026-02-01"), pd.Timestamp("2026-03-01")])].copy()
revenue_plot["Месяц"] = revenue_plot["month"].dt.strftime("%B")
revenue_plot["Выручка, млн ₽"] = revenue_plot["revenue"] / 1_000_000
fig, ax = plt.subplots(figsize=(8, 4.5))
rev_bars = ax.bar(revenue_plot["Месяц"], revenue_plot["Выручка, млн ₽"])
ax.bar_label(rev_bars, labels=[f"{value:.2f} млн ₽" for value in revenue_plot["Выручка, млн ₽"]], padding=4)
ax.set_title("Фактическая выручка по успешным платежам")
ax.set_ylabel("Выручка, млн ₽")
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


**Итог второго исследования.** Гипотеза о том, что повышение цен внесло положительный вклад в рост выручки марта 2026 года, **поддерживается проведённым расчётным анализом**: выручка выросла на 33,86%, цены двух тарифов повысились, а сценарий с февральскими ценами даёт меньшую сумму, чем фактический март. В исходном расчёте сценарная разница составляла около **8,96 млн ₽**. Число следует трактовать как оценку при заданных допущениях, а не как точно установленный причинный эффект.

### Итог второй гипотезы

Гипотеза о положительном вкладе повышения цен **поддерживается расчётным анализом**. Выручка выросла на **14,73 млн ₽ (+33,86%)**; вклад средней суммы платежа (**+8,43 млн ₽**) оказался больше вклада количества успешных платежей (**+6,29 млн ₽**). История тарифов подтверждает повышение цен с марта, а контрфактический сценарий при февральских ценах оценивает мартовскую сумму примерно в **49,27 млн ₽** вместо фактических **58,23 млн ₽**.


In [ ]:
# Итоговый вывод по общей симметричной декомпозиции: используем значения, рассчитанные выше.
print(f"Прирост выручки: {delta_revenue:,.2f} ₽")
print(f"Вклад количества успешных платежей: {volume_effect:,.2f} ₽")
print(f"Вклад средней суммы платежа: {average_effect:,.2f} ₽")
print(f"Вклад средней суммы больше вклада количества: {average_effect > volume_effect}")
if average_effect > volume_effect:
    print("\nВ рамках арифметической декомпозиции вклад изменения средней суммы платежа больше вклада количества успешных платежей.")
else:
    print("\nВ рамках арифметической декомпозиции вклад изменения средней суммы платежа не превышает вклад количества успешных платежей.")

print("Ограничение: эта декомпозиция показывает арифметические вклады, но сама по себе не доказывает причинность.")


## Общий вывод и рекомендации

### Ответ на первый бизнес-вопрос

Доля завершений подписок без дополнительных участников составила **21,14%**, а с дополнительными участниками — **21,10%**. Разница около **0,04 процентного пункта** очень мала; `p-value = 0,8407` больше уровня значимости 0,05. Поэтому первая гипотеза **не получила статистического подтверждения**. На основании этих данных нельзя рекомендовать совместное использование как доказанный способ снизить завершение подписок.

### Ответ на второй бизнес-вопрос

Сначала при самостоятельной агрегации успешных платежей по месяцам был обнаружен рост выручки с **43,50 млн ₽** в феврале до **58,23 млн ₽** в марте 2026 года: **+14,73 млн ₽ (+33,86%)**. Симметричная факторная декомпозиция распределила этот прирост между количеством успешных платежей (**+6,29 млн ₽; 42,7%**) и средней суммой платежа (**+8,43 млн ₽; 57,3%**). Значит, в арифметическом разложении больший вклад внесло изменение средней суммы платежа.

История тарифов показывает повышение цены с марта: базовый тариф — **199 → 249 ₽**, расширенный — **399 → 449 ₽**. В исходном контрфактическом расчёте сценарий мартовских платежей при февральских ценах оценивался примерно в **49,27 млн ₽** против фактических **58,23 млн ₽**. Разница около **8,96 млн ₽** поддерживает гипотезу о положительном вкладе повышения цен.

**Итог:** в рамках проведённого расчётного анализа вторая гипотеза **поддерживается**, но нельзя утверждать, что повышение цен единолично вызвало весь рост. Средняя сумма платежа не тождественна тарифной цене, а контрфактический сценарий зависит от корректного сопоставления платежей с историей тарифов и своих допущений.

### Что рекомендуется бизнесу

## Что рекомендуется бизнесу

1. **Ежемесячно сравнивать выручку, количество успешных платежей и среднюю сумму платежа.** Это поможет понимать, за счёт чего растёт или падает выручка, и вовремя замечать негативные изменения.

2. **Привести историю тарифных цен в порядок.** В данных обнаружены пересекающиеся интервалы действия цен. Из-за этого для одной даты могут подходить разные цены, что мешает правильно оценить выручку при старых тарифах. Проверка истории позволит сделать расчёты надёжнее.

3. **Проверять последствия повышения цен в следующие месяцы.** В марте выручка выросла, однако одного сравнения февраля и марта недостаточно, чтобы доказать, что причиной стало именно повышение цен. Наблюдение за следующими месяцами поможет понять, сохраняется ли рост выручки и не сопровождается ли он снижением количества успешных платежей.

4. **Анализировать базовый и расширенный тарифы отдельно.** Декомпозиция показала, что факторы роста выручки различаются: на базовом тарифе больший вклад внесла средняя сумма платежа, а на расширенном — количество успешных платежей. Раздельный анализ поможет бизнесу точнее определять причины изменений и выбирать подходящие действия для каждого тарифа.

